# Fase 00: Obtención de Datos RAW
**Objetivo:** Descargar el dataset sintético de pacientes desde Kaggle, extraer los 5 CSVs y persistirlos directamente en la capa RAW de Azure Data Lake.
**Entradas:** Dataset ZIP desde API de Kaggle (autenticación via Key Vault).
**Salidas:** Archivos CSV en `medicalproyect-raw/`.

## 1. Inicialización y Configuración

In [ ]:
# ── Función helper: sube el log al Data Lake ──
def _subir_log(sufijo, storage_account):
    try:
        from notebookutils import mssparkutils
        import os
        from datetime import datetime
        fecha = datetime.now().strftime('%Y%m%d_%H%M%S')
        ruta_local = "file://" + os.path.abspath(log_filename)
        ruta_destino = f"abfss://medicalproyect-logs@{storage_account}.dfs.core.windows.net/{sufijo}.log"
        mssparkutils.fs.cp(ruta_local, ruta_destino)
    except Exception:
        pass


# ── Función helper para notificaciones por Telegram ──
def _enviar_telegram(mensaje):
    try:
        from notebookutils import mssparkutils
        import requests
        token = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramToken", NOMBRE_PUENTE)
        chat_id = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "TelegramChatId", NOMBRE_PUENTE)
        url = f"https://api.telegram.org/bot{token}/sendMessage"
        requests.post(url, json={"chat_id": chat_id, "text": mensaje, "parse_mode": "HTML"}, timeout=10)
    except Exception:
        pass

# ── Configuración de cuenta de almacenamiento ──
STORAGE_ACCOUNT = "stproyectomastergrupo3"
CONTAINER_RAW = "medicalproyect-raw"

# ── Importamos librerías para descarga, ZIP y Azure ──
import requests
import zipfile
import io
import datetime
import logging
from notebookutils import mssparkutils

# ── Configuración de Key Vault ──
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = "AzureKeyVault"


# ── Configuración de logging ──
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

log_filename = "pipeline_obtencion_raw.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_filename, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("Medical.ObtencionRAW")

logger.info("=" * 60)
logger.info("INICIO DEL PROCESO DE OBTENCIÓN DE DATOS RAW")
logger.info(f"Timestamp: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
logger.info("=" * 60)
logger.info(f"Storage account: {STORAGE_ACCOUNT}")
logger.info(f"Contenedor RAW:  {CONTAINER_RAW}")
logger.info("📱 Enviando notificación de inicio...")
_enviar_telegram("🚀 INICIADO: 00 Obtencion Datos Raw")


## 2. Configuración de Key Vault y Autenticación Kaggle

In [ ]:
# ── Variables de conexión ──
# El Key Vault almacena las credenciales de Kaggle de forma segura
NOMBRE_BOVEDA = "kv-medicalproyect"
NOMBRE_PUENTE = NOMBRE_PUENTE  # Nombre del servicio vinculado en Synapse
URL_DATASET = "https://www.kaggle.com/api/v1/datasets/download/sergionefedov/patient-records-100k-patients-15-conditions"

try:
    # Obtenemos las credenciales desde Azure Key Vault
    logger.info("Obteniendo credenciales de Kaggle desde Key Vault...")
    kaggle_user = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "KaggleUser", NOMBRE_PUENTE)
    kaggle_key = mssparkutils.credentials.getSecret(NOMBRE_BOVEDA, "KaggleApiToken", NOMBRE_PUENTE)
    logger.info("Credenciales obtenidas correctamente.")
except Exception as e:
    logger.error(f"Error al obtener credenciales de Key Vault: {e}")
    logger.error("Verifica que el Key Vault y el servicio vinculado estén configurados.")
    raise

## 3. Descarga del Dataset desde Kaggle

In [ ]:
logger.info("Descargando dataset desde Kaggle...")

# Descargamos el ZIP completo en memoria (evita escribir archivos temporales)
respuesta = requests.get(URL_DATASET, auth=(kaggle_user, kaggle_key))
respuesta.raise_for_status()

logger.info(f"ZIP descargado correctamente ({len(respuesta.content):,} bytes).")

## 4. Extracción y Persistencia en Capa RAW

In [ ]:
# ── Extraemos el ZIP en memoria y subimos cada CSV a Azure ──
RUTA_RAW = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
archivo_zip = zipfile.ZipFile(io.BytesIO(respuesta.content))
archivos_en_zip = archivo_zip.namelist()

logger.info(f"Archivos encontrados en el ZIP: {archivos_en_zip}")

archivos_subidos = 0
for nombre_archivo in archivos_en_zip:
    # Ignoramos archivos del sistema (MACOSX, etc.)
    if nombre_archivo.startswith('__') or nombre_archivo.startswith('.'):
        continue

    logger.info(f"Subiendo {nombre_archivo} a {RUTA_RAW}/...")
    contenido_csv = archivo_zip.read(nombre_archivo).decode('utf-8')
    ruta_destino = f"{RUTA_RAW}/{nombre_archivo}"
    mssparkutils.fs.put(ruta_destino, contenido_csv, True)
    archivos_subidos += 1

logger.info(f"Subida completada: {archivos_subidos} archivos enviados a {RUTA_RAW}/")

## 5. Validación y Cierre

In [ ]:
# ── Verificamos que los 5 archivos esperados están en el Data Lake ──
ARCHIVOS_ESPERADOS = ["patients.csv", "outcomes.csv", "medications.csv", "lab_results.csv", "diagnoses.csv"]

try:
    archivos_en_datalake = mssparkutils.fs.ls(RUTA_RAW)
    nombres_encontrados = [f.name for f in archivos_en_datalake]

    for archivo in ARCHIVOS_ESPERADOS:
        if archivo in nombres_encontrados:
            logger.info(f"  ✅ {archivo} encontrado en RAW")
        else:
            logger.warning(f"  ⚠️ {archivo} NO encontrado en RAW")

    logger.info("EXECUTION STATUS: SUCCESS")
    _enviar_telegram("✅ FINALIZADO: 00 Obtencion Datos Raw")
except Exception as e:
    logger.error(f"Error al validar archivos en RAW: {e}")
    logger.error("EXECUTION STATUS: FAILED")
    raise
finally:
    # Subimos el log al Data Lake aunque haya fallado
    _subir_log("raw/obtencion_datos", STORAGE_ACCOUNT)

    
spark.stop()